In [1]:
# One-time deps (skip if already in .venv)
%pip install -q -r "d:/HCMUS_ComputerScience/code/AIC2026/requirements.txt"


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [5]:
# Local Whisper JSONL roots (rglob *.jsonl). Add more Lxx paths as needed.
ASR_DIRS = [
    # r"C:/AIC2026-media/ASR/L21",
    r"C:/AIC2026-media/ASR/L22-23-24/L22",
    r"C:/AIC2026-media/ASR/L22-23-24/L23",
    r"C:/AIC2026-media/ASR/L22-23-24/L24",
]


In [6]:
# Output: repo features/asr_emb/Lxx/VIDEO_ID.{npy,jsonl} + model.json (L21–L24 working set)
import json
import re
from pathlib import Path

REPO = Path(r"d:/HCMUS_ComputerScience/code/AIC2026").resolve()
OUT_ROOT = REPO / "features" / "asr_emb"
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
DEVICE = "cpu"  # cuda | cpu
BATCH_SIZE = 8
BATCH_RE = re.compile(r"^(L\d+)_", re.I)

missing = [d for d in ASR_DIRS if not Path(d).is_dir()]
if missing:
    raise SystemExit(f"Not a directory: {missing}")

jsonl_files = []
seen = set()
for d in ASR_DIRS:
    for p in sorted(Path(d).rglob("*.jsonl")):
        key = p.stem
        if key in seen:
            print(f"warning: duplicate stem {key}, keep first", flush=True)
            continue
        seen.add(key)
        jsonl_files.append(p)

if not jsonl_files:
    raise SystemExit(f"No .jsonl under ASR_DIRS={ASR_DIRS}")

print(f"jsonl_count={len(jsonl_files)}")
print("REPO =", REPO)
print("OUT_ROOT =", OUT_ROOT)
print("MODEL_NAME =", MODEL_NAME)
print("DEVICE =", DEVICE)


jsonl_count=99
REPO = D:\HCMUS_ComputerScience\code\AIC2026
OUT_ROOT = D:\HCMUS_ComputerScience\code\AIC2026\features\asr_emb
MODEL_NAME = sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
DEVICE = cpu


In [7]:
# Segment embed: skip empty text; write filtered jsonl + L2-normalized .npy (row i ↔ line i).
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

out_root = Path(OUT_ROOT)
out_root.mkdir(parents=True, exist_ok=True)

device = DEVICE if (DEVICE == "cpu" or torch.cuda.is_available()) else "cpu"
if DEVICE == "cuda" and device == "cpu":
    print("warning: CUDA requested but unavailable; using cpu", flush=True)

model = SentenceTransformer(MODEL_NAME, device=device)
dim = int(model.get_embedding_dimension())
print(f"model loaded dim={dim} device={device}", flush=True)


def load_segments(path: Path) -> list[dict]:
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        text = str(obj.get("text") or "").strip()
        if not text:
            continue
        rows.append(
            {
                "start": float(obj["start"]),
                "end": float(obj["end"]),
                "text": text,
            }
        )
    return rows


def batch_id(stem: str) -> str:
    m = BATCH_RE.match(stem)
    return m.group(1) if m else "unknown"


n_ok = 0
n_empty = 0
for i, src in enumerate(jsonl_files, start=1):
    stem = src.stem
    batch = batch_id(stem)
    dest_dir = out_root / batch
    dest_dir.mkdir(parents=True, exist_ok=True)
    npy_path = dest_dir / f"{stem}.npy"
    jsonl_path = dest_dir / f"{stem}.jsonl"

    segs = load_segments(src)
    if not segs:
        n_empty += 1
        print(f"[{i}/{len(jsonl_files)}] skip empty {stem}", flush=True)
        continue

    texts = [s["text"] for s in segs]
    emb = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32, copy=False)

    if emb.ndim != 2 or emb.shape[0] != len(segs):
        raise SystemExit(f"Bad embed shape {emb.shape} for {stem} n={len(segs)}")

    np.save(npy_path, emb)
    with jsonl_path.open("w", encoding="utf-8") as f:
        for s in segs:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")

    n_ok += 1
    if i == 1 or i % 10 == 0 or i == len(jsonl_files):
        print(
            f"[{i}/{len(jsonl_files)}] {batch}/{stem}  segs={len(segs)}  → {npy_path}",
            flush=True,
        )

meta = {"model": MODEL_NAME, "dim": dim, "normalize": True}
(out_root / "model.json").write_text(
    json.dumps(meta, indent=2) + "\n", encoding="utf-8"
)
print(f"Done: wrote={n_ok}  empty_skipped={n_empty}  dim={dim}")
print("model.json =", out_root / "model.json")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3100.67it/s]


model loaded dim=384 device=cpu
[1/99] L22/L22_V001  segs=430  → D:\HCMUS_ComputerScience\code\AIC2026\features\asr_emb\L22\L22_V001.npy
[10/99] L22/L22_V010  segs=210  → D:\HCMUS_ComputerScience\code\AIC2026\features\asr_emb\L22\L22_V010.npy
[20/99] L22/L22_V020  segs=527  → D:\HCMUS_ComputerScience\code\AIC2026\features\asr_emb\L22\L22_V020.npy
[30/99] L22/L22_V030  segs=492  → D:\HCMUS_ComputerScience\code\AIC2026\features\asr_emb\L22\L22_V030.npy
[40/99] L23/L23_V009  segs=51  → D:\HCMUS_ComputerScience\code\AIC2026\features\asr_emb\L23\L23_V009.npy
[50/99] L23/L23_V019  segs=170  → D:\HCMUS_ComputerScience\code\AIC2026\features\asr_emb\L23\L23_V019.npy
[60/99] L24/L24_V005  segs=1  → D:\HCMUS_ComputerScience\code\AIC2026\features\asr_emb\L24\L24_V005.npy
[63/99] skip empty L24_V008
[68/99] skip empty L24_V013
[70/99] skip empty L24_V015
[71/99] skip empty L24_V016
[74/99] skip empty L24_V019
[76/99] skip empty L24_V021
[80/99] L24/L24_V025  segs=1  → D:\HCMUS_ComputerScience\code\